[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_01/MFM_ch1_path_dependent_vol/MFM_ch1_path_dependent_vol.ipynb)

# MFM_ch1_path_dependent_vol

Case study, Guyon and Lekeufack (2023), Volatility is (mostly) path-dependent: the VIX as a linear function of a kernel-weighted trend and a kernel-weighted volatility of past S&P 500 returns, eq. (3.7), with normalised time-shifted power-law kernels, eq. (3.9), 1000 past returns, estimated by nonlinear least squares on 2000-2018 from the VIX row of Table 3; r2 and RMSE on the training set, the test set 2019 - 15 May 2022 and after the test set with frozen parameters; observed versus predicted VIX, their ratio, and the fitted kernels.

*Modelling Financial Markets (MFM), Chapter 1: Stylised Facts and Returns. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/MFM/Chapter_1'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.tsa.stattools import acf
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
import warnings
warnings.filterwarnings('ignore')

## Style and helpers

In [ ]:
# Stil standard MFM (identic cu SFM): transparent + ENG + legenda jos
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Culori brand
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Gray     = '#7F7F7F'
LightGray = '#DADADA'
Teal     = '#17A2B8'

ASSETS = ['sp500', 'bettr', 'btc', 'eurron', 'gold']
COLORS = {'sp500': MainBlue, 'bettr': IDAred, 'btc': Amber, 'eurron': Forest, 'gold': Purple}
PERIODS = {'sp500': 252, 'bettr': 252, 'btc': 365, 'eurron': 252, 'gold': 252}


def save_fig(name):
    """Salveaza figura ca PDF si PNG transparent, in folderul curent."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Plaseaza legenda in afara graficului, jos-centru."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
# copia locala a folderului data/market, daca notebook-ul ruleaza din repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')

SOURCES = {
    'sp500':  ('GSPC.INDX',    'close', '1990-01-01'),
    'bettr':  ('BETTR.INDX',   'close', '2014-09-01'),
    'btc':    ('BTC-USD.CC',   'close', '2014-09-17'),
    'eurron': ('REF:EUR',      'close', '2005-07-01'),   # de la introducerea leului nou (RON), 1 iulie 2005
    'gold':   ('XAUUSD.FOREX', 'close', '1990-01-01'),
}

LABELS = {'sp500': 'S&P 500', 'bettr': 'BET-TR (Romania)', 'btc': 'Bitcoin',
          'eurron': 'EUR/RON', 'gold': 'Gold'}

# cotatii OTC (aur): sesiunile partiale de sambata/duminica rup randamentul vineri -> luni,
# deci pastram doar zilele lucratoare (aceeasi conventie ca in Capitolul 0)
WEEKDAYS_ONLY = {'gold'}

_CACHE = {}


def read_market(symbol):
    """Citeste data/market/<SIMBOL>.csv local sau din repo-ul GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname)
    src = path if os.path.exists(path) else REPO_RAW + fname
    df = pd.read_csv(src, parse_dates=['date']).set_index('date').sort_index()
    df.index.name = 'Date'
    return df


def read_reference_rate(currency='EUR', start='2005-07-01', end='2026-09-18'):
    """Cursul oficial de referinta RON (arhive XML anuale BNR)."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    import re
    import urllib.request
    rows = []
    for y in range(int(start[:4]), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}">([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    df = pd.DataFrame(rows, columns=['Date', 'Close']).drop_duplicates('Date').set_index('Date')
    df.index = pd.to_datetime(df.index)
    _CACHE[key] = df.sort_index().loc[start:end]
    return _CACHE[key]


def load_data(name='sp500'):
    """OHLCV zilnic cu coloane Open/High/Low/Close/Volume (Close = pretul folosit)."""
    symbol, col, start = SOURCES[name]
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start)
    d = read_market(symbol).loc[start:]
    if name in WEEKDAYS_ONLY:
        d = d[d.index.dayofweek < 5]
    out = pd.DataFrame({'Open': d['open'], 'High': d['high'], 'Low': d['low'],
                        'Close': d[col], 'Volume': d['volume']})
    return out[out['Close'] > 0].dropna(subset=['Close'])


def load_close(name):
    """Pretul de inchidere, folosit in toate graficele."""
    return load_data(name)['Close'].dropna()


closes = {a: load_close(a) for a in ASSETS}
rets = {a: np.log(c).diff().dropna() for a, c in closes.items()}
# gold (weekdays only) is annualised with its actual number of observations per year
PERIODS['gold'] = len(rets['gold']) / ((closes['gold'].index[-1] - closes['gold'].index[0]).days / 365.25)
spx_ohlc = load_data('sp500')
for a in ASSETS:
    print(f'{LABELS[a]:18s}', rets[a].index[0].date(), '->', rets[a].index[-1].date(), len(rets[a]), 'obs')

## Helper functions

In [ ]:
PDV_LAGS = 1000

PDV_DT = 1 / 252

PDV_TABLE3_VIX = (0.057, -0.095, 0.82, 1.06, 0.020, 1.60, 0.052)   # b0, b1, b2, a1, d1, a2, d2 (Table 3)

PDV_TRAIN = ('2000-01-01', '2018-12-31')

PDV_TEST = ('2019-01-01', '2022-05-15')

def pdv_kernel(alpha, delta, n=PDV_LAGS):
    """Nucleul TSPL normalizat, eq. (3.9): K(tau) = (tau + delta)^(-alpha) / Z, tau = i * dt."""
    tau = np.arange(n) * PDV_DT
    return (tau + delta) ** (-alpha) * (alpha - 1) / delta ** (1 - alpha)


def pdv_predict(X, p):
    """Modelul (3.7): sigma_t = b0 + b1 R1_t + b2 Sigma_t; X[t, i] = r_{t-i}."""
    b0, b1, b2, a1, d1, a2, d2 = p
    R1 = X @ pdv_kernel(a1, d1)
    Sig = np.sqrt((X ** 2) @ pdv_kernel(a2, d2))
    return b0 + b1 * R1 + b2 * Sig


def pdv_data():
    """Matricea randamentelor simple trecute ale S&P 500 si VIX/100 pe zilele S&P 500."""
    spx = read_market('GSPC.INDX')['close']
    r = spx / spx.shift(1) - 1
    X = pd.DataFrame(np.column_stack([r.shift(i).values for i in range(PDV_LAGS)]), index=r.index)
    vix = read_market('VIX.INDX')['close'] / 100
    return X.join(vix.rename('vix'), how='inner').dropna()

## Analysis

In [ ]:
def pdv_fit():
    """Celor mai mici patrate neliniare pe 2000-2018, pornind de la randul VIX din Table 3."""
    from scipy.optimize import least_squares
    df = pdv_data()
    tr = df.loc[PDV_TRAIN[0]:PDV_TRAIN[1]]
    lb = [-np.inf] * 3 + [1.0001, 1e-4, 1.0001, 1e-4]
    ub = [np.inf] * 3 + [20, 5, 20, 5]
    res = least_squares(lambda p: pdv_predict(tr.iloc[:, :PDV_LAGS].values, p) - tr['vix'].values,
                        x0=PDV_TABLE3_VIX, bounds=(lb, ub), method='trf')
    fit = pd.Series(pdv_predict(df.iloc[:, :PDV_LAGS].values, res.x), index=df.index)
    rows = []
    for name, (a, b) in [('train', PDV_TRAIN), ('test', PDV_TEST),
                         ('after test', ('2022-05-16', str(df.index[-1].date())))]:
        y, yh = df['vix'].loc[a:b], fit.loc[a:b]
        rows.append({'set': name, 'start': y.index[0].date(), 'end': y.index[-1].date(), 'N': len(y),
                     'RMSE': np.sqrt(np.mean((y - yh) ** 2)),
                     'r2': 1 - np.sum((y - yh) ** 2) / np.sum((y - y.mean()) ** 2)})
    scores = pd.DataFrame(rows).set_index('set')
    params = pd.Series(res.x, index=['beta0', 'beta1', 'beta2', 'alpha1', 'delta1', 'alpha2', 'delta2'])
    params.to_csv(os.path.join(TABLE_DIR, 'ch1_pdv_parameters.csv'), float_format='%.4g')
    scores.to_csv(os.path.join(TABLE_DIR, 'ch1_pdv_scores.csv'), float_format='%.4g')
    return df['vix'], fit, params, scores


def fig_cs_pdv_vix(vix, fit, scores):
    """Figure 3.2 (top) replicat si extins: VIX observat vs. VIX prezis doar din traiectoria S&P 500."""
    fig, (ax, ax2) = plt.subplots(2, 1, figsize=(7.6, 3.7), sharex=True, gridspec_kw={'height_ratios': [2.3, 1]})
    v, f = 100 * vix.loc[PDV_TRAIN[0]:], 100 * fit.loc[PDV_TRAIN[0]:]
    ax.plot(v.index, v, color=MainBlue, lw=0.7, label='VIX (observed)')
    ax.plot(f.index, f, color=IDAred, lw=0.7, label='Predicted from past S&P 500 returns, eq. (3.7)')
    ratio = vix.loc[PDV_TRAIN[0]:] / fit.loc[PDV_TRAIN[0]:]
    ax2.plot(ratio.index, ratio, color=Forest, lw=0.6, label='Ratio observed / predicted VIX')
    ax2.axhline(1, color=Gray, lw=0.6, ls='--')
    for a in (ax, ax2):
        a.axvspan(pd.Timestamp(PDV_TEST[0]), pd.Timestamp(PDV_TEST[1]), color=Amber, alpha=0.15, lw=0)
        a.axvline(pd.Timestamp(PDV_TEST[1]), color=Gray, lw=0.6, ls=':')
    top = 0.97
    for (a, b), key, lab in [(PDV_TRAIN, 'train', 'Training 2000-2018'), (PDV_TEST, 'test', 'Test'),
                             (('2022-05-16', str(vix.index[-1].date())), 'after test', 'After test')]:
        mid = pd.Timestamp(a) + (pd.Timestamp(b) - pd.Timestamp(a)) / 2
        ax.text(mid, 99, f"{lab}\n$r^2$ = {scores.loc[key, 'r2']:.3f}", ha='center', va='top', fontsize=8, color='black')
    ax.set_ylim(0, 100)
    ax.set_ylabel('Volatility (%)')
    ax2.set_ylabel('Ratio')
    ax2.set_ylim(0.4, 1.8)
    ax.set_title('VIX explained by the S&P 500 path alone (test set shaded)')
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    fig.legend(h1 + h2, l1 + l2, loc='upper center', bbox_to_anchor=(0.5, 0.0), ncol=2, frameon=False)
    plt.tight_layout()
    save_fig('ch1_cs_pdv_vix')


def fig_cs_pdv_kernels(params):
    """Figure 3.1 / B.1 replicat: ponderile zilnice K(i dt) dt ale celor doi nuclei, scala log-log."""
    lag = np.arange(1, PDV_LAGS)
    p3 = PDV_TABLE3_VIX
    fig, ax = plt.subplots(figsize=(6.4, 3.2))
    ax.loglog(lag, pdv_kernel(params['alpha1'], params['delta1'])[1:] * PDV_DT, color=IDAred, lw=1.4,
              label=f"$K_1$ (trend $R_1$), ours: $\\alpha_1$ = {params['alpha1']:.2f}, $\\delta_1$ = {params['delta1']:.3f}")
    ax.loglog(lag, pdv_kernel(p3[3], p3[4])[1:] * PDV_DT, color=Orange, lw=1.2, ls='--',
              label=f"$K_1$, Table 3: $\\alpha_1$ = {p3[3]:.2f}, $\\delta_1$ = {p3[4]:.3f}")
    ax.loglog(lag, pdv_kernel(params['alpha2'], params['delta2'])[1:] * PDV_DT, color=MainBlue, lw=1.4,
              label=f"$K_2$ (volatility $\\Sigma$), ours: $\\alpha_2$ = {params['alpha2']:.2f}, $\\delta_2$ = {params['delta2']:.3f}")
    ax.loglog(lag, pdv_kernel(p3[5], p3[6])[1:] * PDV_DT, color=Teal, lw=1.4, ls=(0, (2, 2)),
              label=f"$K_2$, Table 3: $\\alpha_2$ = {p3[5]:.2f}, $\\delta_2$ = {p3[6]:.3f}")
    ax.set_xlabel('Lag i (trading days, log scale)')
    ax.set_ylabel('Weight $K(i\\Delta t)\\Delta t$ (log scale)')
    ax.set_title('Fitted kernels of the VIX model')
    legend_outside_bottom(ax, ncol=2, y=-0.2)
    plt.tight_layout()
    save_fig('ch1_cs_pdv_kernels')


def pdv_weight_shares(params):
    """Ponderea totala (din 1000 de zile) pe ultimele 5 zile si dincolo de 250 de zile."""
    out = {}
    for k, (a, d) in {'K1': (params['alpha1'], params['delta1']), 'K2': (params['alpha2'], params['delta2'])}.items():
        w = pdv_kernel(a, d) * PDV_DT
        out[k] = {'today': w[0], 'first5': w[:5].sum(), 'beyond250': w[250:].sum(), 'total1000': w.sum()}
    return pd.DataFrame(out)

## Run

In [ ]:
TABLE_DIR = '.'
vix, fit, params, scores = pdv_fit()
print(params.round(3))
print(scores.round(3))
fig_cs_pdv_vix(vix, fit, scores)
fig_cs_pdv_kernels(params)
print(pdv_weight_shares(params).round(3))

![ch1_cs_pdv_vix](./ch1_cs_pdv_vix.png)

Output: `ch1_cs_pdv_vix.pdf`

![ch1_cs_pdv_kernels](./ch1_cs_pdv_kernels.png)

Output: `ch1_cs_pdv_kernels.pdf`